In [1]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [3]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score
from sklearn.pipeline import Pipeline
from scipy.sparse import hstack
import warnings
warnings.filterwarnings('ignore')

train = pd.read_csv('train.csv')
test  = pd.read_csv('test.csv')
sub   = pd.read_csv('sample_submission.csv')

print("Train shape:", train.shape)
print("Test shape :", test.shape)
print("\nTarget distribution:")
print(train['rule_violation'].value_counts(normalize=True).round(4))

Train shape: (2029, 9)
Test shape : (10, 8)

Target distribution:
rule_violation
1    0.5081
0    0.4919
Name: proportion, dtype: float64


In [5]:
def build_text_features(df):
    """
    Gabungkan semua teks yang relevan jadi satu representasi.
    Key insight: body + rule + contoh positif/negatif semuanya mengandung signal.
    """
    df = df.copy()

    # Gabung body + rule — inti dari task ini
    df['body_rule'] = df['body'].fillna('') + ' [SEP] ' + df['rule'].fillna('')

    # Full context: body + rule + semua contoh
    df['full_context'] = (
        df['body'].fillna('') + ' [SEP] ' +
        df['rule'].fillna('') + ' [SEP] ' +
        df['positive_example_1'].fillna('') + ' [SEP] ' +
        df['positive_example_2'].fillna('') + ' [SEP] ' +
        df['negative_example_1'].fillna('') + ' [SEP] ' +
        df['negative_example_2'].fillna('')
    )

    # Similarity signal: apakah body mirip dengan positive atau negative example?
    # Hitung overlap kata (Jaccard-like)
    def word_overlap(text1, text2):
        words1 = set(str(text1).lower().split())
        words2 = set(str(text2).lower().split())
        if not words1 or not words2:
            return 0.0
        return len(words1 & words2) / len(words1 | words2)

    df['overlap_pos1'] = df.apply(lambda r: word_overlap(r['body'], r['positive_example_1']), axis=1)
    df['overlap_pos2'] = df.apply(lambda r: word_overlap(r['body'], r['positive_example_2']), axis=1)
    df['overlap_neg1'] = df.apply(lambda r: word_overlap(r['body'], r['negative_example_1']), axis=1)
    df['overlap_neg2'] = df.apply(lambda r: word_overlap(r['body'], r['negative_example_2']), axis=1)

    # Max overlap — seberapa mirip body dengan contoh pelanggaran
    df['max_overlap_pos'] = df[['overlap_pos1', 'overlap_pos2']].max(axis=1)
    df['max_overlap_neg'] = df[['overlap_neg1', 'overlap_neg2']].max(axis=1)

    # Violation signal: lebih mirip positive (contoh pelanggaran) = lebih mungkin melanggar
    df['violation_signal'] = df['max_overlap_pos'] - df['max_overlap_neg']

    # Text length features
    df['body_len']      = df['body'].str.len().fillna(0)
    df['body_words']    = df['body'].str.split().str.len().fillna(0)
    df['rule_len']      = df['rule'].str.len().fillna(0)

    return df

train = build_text_features(train)
test  = build_text_features(test)
print("Features built!")
print(train[['overlap_pos1','overlap_neg1','violation_signal']].describe())

Features built!
       overlap_pos1  overlap_neg1  violation_signal
count   2029.000000   2029.000000       2029.000000
mean       0.045105      0.037651          0.007327
std        0.065690      0.063773          0.106758
min        0.000000      0.000000         -1.000000
25%        0.000000      0.000000         -0.013416
50%        0.037736      0.027778          0.002381
75%        0.068966      0.059701          0.033311
max        1.000000      1.000000          1.000000


In [6]:
# TF-IDF pada body saja
tfidf_body = TfidfVectorizer(
    max_features=5000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{1,}',
    stop_words='english'
)

# TF-IDF pada full context (body + rule + examples)
tfidf_full = TfidfVectorizer(
    max_features=3000,
    ngram_range=(1, 2),
    min_df=2,
    sublinear_tf=True,
    strip_accents='unicode',
    analyzer='word',
    token_pattern=r'\w{1,}',
    stop_words='english'
)

# Fit pada combined train+test untuk coverage lebih baik
all_body    = pd.concat([train['body_rule'], test['body_rule']])
all_context = pd.concat([train['full_context'], test['full_context']])

tfidf_body.fit(all_body)
tfidf_full.fit(all_context)

# Transform
X_body_tr   = tfidf_body.transform(train['body_rule'])
X_full_tr   = tfidf_full.transform(train['full_context'])
X_body_te   = tfidf_body.transform(test['body_rule'])
X_full_te   = tfidf_full.transform(test['full_context'])

# Numeric features
numeric_cols = [
    'overlap_pos1', 'overlap_pos2', 'overlap_neg1', 'overlap_neg2',
    'max_overlap_pos', 'max_overlap_neg', 'violation_signal',
    'body_len', 'body_words', 'rule_len'
]

from scipy.sparse import csr_matrix
X_num_tr = csr_matrix(train[numeric_cols].values)
X_num_te = csr_matrix(test[numeric_cols].values)

# Stack semua features
X_train = hstack([X_body_tr, X_full_tr, X_num_tr])
X_test  = hstack([X_body_te, X_full_te, X_num_te])
y_train = train['rule_violation'].values

print(f"X_train shape: {X_train.shape}")
print(f"X_test shape : {X_test.shape}")

X_train shape: (2029, 8010)
X_test shape : (10, 8010)


In [7]:
# Model 1: Logistic Regression — baseline kuat untuk NLP
lr = LogisticRegression(C=1.0, max_iter=1000, random_state=42)

# CV dengan StratifiedKFold
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_scores = cross_val_score(lr, X_train, y_train, cv=skf, scoring='roc_auc')

print("="*55)
print("  Logistic Regression — 5-Fold CV")
print("="*55)
for i, score in enumerate(cv_scores, 1):
    print(f"  Fold {i} | AUC: {score:.5f}")
print(f"  Mean AUC : {cv_scores.mean():.5f} ± {cv_scores.std():.5f}")
print("="*55)

# Train pada full data
lr.fit(X_train, y_train)
test_preds = lr.predict_proba(X_test)[:, 1]

print(f"\nTest predictions:")
for i, (idx, pred) in enumerate(zip(test['row_id'].values, test_preds)):
    print(f"  row_id {idx}: {pred:.4f}")

  Logistic Regression — 5-Fold CV
  Fold 1 | AUC: 0.82752
  Fold 2 | AUC: 0.76146
  Fold 3 | AUC: 0.79546
  Fold 4 | AUC: 0.80383
  Fold 5 | AUC: 0.77641
  Mean AUC : 0.79293 ± 0.02274

Test predictions:
  row_id 2029: 0.1850
  row_id 2030: 0.3683
  row_id 2031: 0.6193
  row_id 2032: 0.6039
  row_id 2033: 0.7448
  row_id 2034: 0.2822
  row_id 2035: 0.8145
  row_id 2036: 0.3193
  row_id 2037: 0.1196
  row_id 2038: 0.7249


In [8]:
sub['rule_violation'] = test_preds
sub.to_csv('submission.csv', index=False)

print("Submission:")
print(sub)

Submission:
   row_id  rule_violation
0    2029        0.184998
1    2030        0.368327
2    2031        0.619257
3    2032        0.603868
4    2033        0.744798
5    2034        0.282195
6    2035        0.814477
7    2036        0.319265
8    2037        0.119568
9    2038        0.724945


In [9]:
# Upgrade: pakai sentence-transformers untuk semantic similarity
# Install dulu (internet harus ON saat development, OFF saat commit)
!pip install sentence-transformers -q

from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model_st = SentenceTransformer('all-MiniLM-L6-v2')

def get_semantic_features(df):
    df = df.copy()

    # Encode semua teks
    body_emb  = model_st.encode(df['body'].fillna('').tolist())
    pos1_emb  = model_st.encode(df['positive_example_1'].fillna('').tolist())
    neg1_emb  = model_st.encode(df['negative_example_1'].fillna('').tolist())
    rule_emb  = model_st.encode(df['rule'].fillna('').tolist())

    # Cosine similarity — seberapa mirip body dengan contoh pelanggaran
    sim_pos = [cosine_similarity([b], [p])[0][0]
               for b, p in zip(body_emb, pos1_emb)]
    sim_neg = [cosine_similarity([b], [n])[0][0]
               for b, n in zip(body_emb, neg1_emb)]
    sim_rule = [cosine_similarity([b], [r])[0][0]
                for b, r in zip(body_emb, rule_emb)]

    df['semantic_sim_pos']  = sim_pos
    df['semantic_sim_neg']  = sim_neg
    df['semantic_sim_rule'] = sim_rule
    df['semantic_signal']   = df['semantic_sim_pos'] - df['semantic_sim_neg']

    return df, body_emb

train_sem, train_emb = get_semantic_features(train)
test_sem,  test_emb  = get_semantic_features(test)
print("Semantic features done!")
print(train_sem[['semantic_sim_pos','semantic_sim_neg','semantic_signal']].describe())

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Semantic features done!
       semantic_sim_pos  semantic_sim_neg  semantic_signal
count       2029.000000       2029.000000      2029.000000
mean           0.137249          0.123622         0.013627
std            0.121778          0.130073         0.173294
min           -0.223875         -0.176172        -1.031968
25%            0.056462          0.041928        -0.077064
50%            0.127587          0.103815         0.015098
75%            0.202202          0.180145         0.108217
max            1.000000          1.000000         1.126637


In [10]:
from scipy.sparse import csr_matrix, hstack

# Gabung TF-IDF + semantic features
sem_cols = ['semantic_sim_pos','semantic_sim_neg',
            'semantic_sim_rule','semantic_signal']

X_sem_tr = csr_matrix(train_sem[sem_cols].values)
X_sem_te = csr_matrix(test_sem[sem_cols].values)

# Stack semua features
X_train_v2 = hstack([X_train, X_sem_tr])
X_test_v2  = hstack([X_test,  X_sem_te])

# Retrain dengan semantic features
lr2 = LogisticRegression(C=1.0, max_iter=1000, random_state=42)
cv2 = cross_val_score(lr2, X_train_v2, y_train, cv=skf, scoring='roc_auc')

print("="*50)
print("  V2 — TF-IDF + Semantic Features")
print("="*50)
for i, s in enumerate(cv2, 1):
    print(f"  Fold {i} | AUC: {s:.5f}")
print(f"  Mean AUC: {cv2.mean():.5f} ± {cv2.std():.5f}")
print("="*50)

# Generate submission v2
lr2.fit(X_train_v2, y_train)
test_preds_v2 = lr2.predict_proba(X_test_v2)[:, 1]

sub['rule_violation'] = test_preds_v2
sub.to_csv('submission_v2.csv', index=False)
print("\nSubmission V2 preview:")
print(sub)

  V2 — TF-IDF + Semantic Features
  Fold 1 | AUC: 0.83301
  Fold 2 | AUC: 0.78828
  Fold 3 | AUC: 0.80743
  Fold 4 | AUC: 0.83043
  Fold 5 | AUC: 0.79014
  Mean AUC: 0.80986 ± 0.01908

Submission V2 preview:
   row_id  rule_violation
0    2029        0.178566
1    2030        0.349675
2    2031        0.643242
3    2032        0.587276
4    2033        0.781221
5    2034        0.211503
6    2035        0.786479
7    2036        0.281975
8    2037        0.130644
9    2038        0.771723
